# 04 CNN Training

Stage 7 prepares PyTorch-ready tensors for deep learning, Stage 8 trains the first validation-monitored 1D CNN, Stage 9 compares basic training choices for that same single-epoch CNN, Stage 10 compares simple and temporal-context CNNs on matched context-eligible center epochs, Stage 11 compares many-to-one CNN-GRU sequence models, Stage 12 trains many-to-many CNN-GRU models with overlapping-window probability aggregation, and Stage 14 trains one multiscale residual CNN fused with engineered epoch features. This notebook starts with shape, leakage, and preprocessing checks, then runs guarded validation-only training workflows through reusable `src.train` utilities. The held-out test split is not evaluated here.


This setup cell imports the reusable data and training utilities, resolves paths whether the notebook is run from the repository root or the `notebooks/` directory, and defines a small debug configuration for the initial Stage 8 smoke run. The expected output is a single Boolean indicating whether local raw data and `data/interim/epoch_index.csv` are available. If it is `False`, later cells skip cleanly instead of failing on missing local DREAMT artifacts. Stage 9, Stage 10, Stage 11, Stage 12, and Stage 14 cells remain guarded so routine execution does not launch long training runs.


In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import torch

from src.error_analysis import class_prior_correction_sweep
from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_FEATURE_PREPROCESSING_METADATA_PATH,
    DEFAULT_PARTICIPANT_ARRAY_CACHE_DIR,
    DEFAULT_PREPROCESSING_METADATA_PATH,
    DEFAULT_RAW_DATA_DIR,
    DEFAULT_TRAIN_FEATURES_PATH,
    DEFAULT_VALIDATION_FEATURES_PATH,
    DreamtContextDataset,
    DreamtEpochDataset,
    DreamtSequenceDataset,
    check_epoch_split_leakage,
    fit_normalization_stats,
    load_preprocessing_metadata,
    save_preprocessing_metadata,
)
from src.train import (
    DEFAULT_STAGE8_OUTPUT_DIR,
    DEFAULT_STAGE9_OUTPUT_DIR,
    DEFAULT_STAGE10_OUTPUT_DIR,
    DEFAULT_STAGE11_OUTPUT_DIR,
    DEFAULT_STAGE11_LOSS_OUTPUT_DIR,
    DEFAULT_STAGE12_OUTPUT_DIR,
    DEFAULT_STAGE14_OUTPUT_DIR,
    TrainConfig,
    build_stage9_screening_configs,
    build_stage10_comparison_configs,
    build_stage11_loss_comparison_configs,
    build_stage11_sequence_configs,
    build_stage12_many_to_many_configs,
    build_stage14_fusion_config,
    build_train_validation_datasets,
    class_counts_from_loader,
    load_train_config,
    run_tiny_overfit_test,
    run_stage9_experiments,
    run_stage10_experiments,
    run_stage11_experiments,
    run_stage12_experiments,
    run_stage14_experiment,
    train_model,
)

CHANNELS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
BATCH_SIZE = 16
DEBUG_PARTICIPANTS = 3
EPOCHS = 1

raw_dir = repo_root / DEFAULT_RAW_DATA_DIR
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
metadata_path = repo_root / DEFAULT_PREPROCESSING_METADATA_PATH
array_cache_dir = repo_root / DEFAULT_PARTICIPANT_ARRAY_CACHE_DIR
train_feature_path = repo_root / DEFAULT_TRAIN_FEATURES_PATH
validation_feature_path = repo_root / DEFAULT_VALIDATION_FEATURES_PATH
feature_metadata_path = repo_root / DEFAULT_FEATURE_PREPROCESSING_METADATA_PATH
output_dir = repo_root / DEFAULT_STAGE8_OUTPUT_DIR
stage9_output_dir = repo_root / DEFAULT_STAGE9_OUTPUT_DIR
stage10_output_dir = repo_root / DEFAULT_STAGE10_OUTPUT_DIR
stage11_output_dir = repo_root / DEFAULT_STAGE11_OUTPUT_DIR
stage11_loss_output_dir = repo_root / DEFAULT_STAGE11_LOSS_OUTPUT_DIR
stage12_output_dir = repo_root / DEFAULT_STAGE12_OUTPUT_DIR
stage14_output_dir = repo_root / DEFAULT_STAGE14_OUTPUT_DIR
stage8_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=output_dir,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

artifacts_available = raw_dir.exists() and epoch_index_path.exists()
stage14_artifacts_available = (
    artifacts_available
    and train_feature_path.exists()
    and validation_feature_path.exists()
)
artifacts_available


True

## Build Single-Epoch Datasets

This section builds the PyTorch-ready single-epoch datasets used by the first CNN. The first cell fits streaming mean imputation and per-channel standardization metadata from training epochs only, then saves that metadata for reuse. Expected output is either no displayed output when local artifacts are present, or a skip message when raw files or the epoch index are absent. Validation and test data are not used to fit preprocessing statistics.

In [2]:
if artifacts_available:
    train_unscaled = DreamtEpochDataset(
        raw_dir=raw_dir,
        epoch_index=epoch_index_path,
        split="train",
        channels=CHANNELS,
        max_participants=DEBUG_PARTICIPANTS,
    )
    stats = fit_normalization_stats(train_unscaled)
    save_preprocessing_metadata(stats, metadata_path)
else:
    print("Skipping dataset construction because local raw files or epoch_index.csv are absent.")


This cell reloads the saved preprocessing metadata, applies it to train and validation datasets, checks that participant-level split leakage is absent within each dataset, and constructs DataLoaders. The printed batch shape should be `(batch, channels, timepoints)` for `x` and one integer label per epoch for `y`; the participant counts confirm the debug subset size; and the metadata channels should match the configured model channels. The test split is intentionally not loaded.

In [3]:
if artifacts_available:
    stats = load_preprocessing_metadata(metadata_path)
    train_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    val_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="validation", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    check_epoch_split_leakage(train_ds.epoch_index)
    check_epoch_split_leakage(val_ds.epoch_index)
    loaders = {
        "train": torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        "validation": torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
    }
    x_batch, y_batch = next(iter(loaders["train"]))
    print("train batch:", tuple(x_batch.shape), x_batch.dtype, tuple(y_batch.shape), y_batch.dtype)
    print("participants:", {"train": len(train_ds.participants), "validation": len(val_ds.participants)})
    print("metadata channels:", stats["channels"])


train batch: (16, 8, 1920) torch.float32 (16,) torch.int64
participants: {'train': 3, 'validation': 3}
metadata channels: ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI']


## Temporal Context And Sequence Shape Checks

This section performs lightweight shape checks for the temporal-context and sequence datasets that will support later CNN-context and CNN-GRU experiments. These checks answer whether neighboring-epoch windows can be formed without crossing participant boundaries and whether the tensor shapes match the intended model families. The expected output is one example context item and one example sequence item when enough consecutive training epochs exist in the debug subset.

In [4]:
if artifacts_available:
    context_ds = DreamtContextDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, context_radius=2, max_participants=DEBUG_PARTICIPANTS)
    sequence_ds = DreamtSequenceDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, sequence_length=5, label_mode="many_to_one", target_position="center", max_participants=DEBUG_PARTICIPANTS)
    if len(context_ds):
        x_context, y_context = context_ds[0]
        print("context item:", tuple(x_context.shape), y_context.item())
    if len(sequence_ds):
        x_sequence, y_sequence = sequence_ds[0]
        print("sequence item:", tuple(x_sequence.shape), y_sequence.item())


context item: (8, 9600) 0
sequence item: (5, 8, 1920) 0


## Tiny Overfit Smoke Test

This cell runs a tiny repeated-batch overfit test on training data only. Its purpose is to verify that the model, loss, optimizer, tensor dtypes, and device handling can reduce training loss before running validation-monitored experiments. The expected output is the first and last loss; the last value should usually be lower than the first. This is a plumbing check, not a model-selection result.

In [5]:
if artifacts_available:
    overfit_history = run_tiny_overfit_test(train_ds, stage8_config)
    print("loss first/last:", round(overfit_history["loss"].iloc[0], 4), round(overfit_history["loss"].iloc[-1], 4))


loss first/last: 1.2609 0.0002


## Single-Epoch CNN Training

This cell trains the first modest single-epoch CNN and monitors validation metrics after each epoch. In the default debug configuration, it runs for one epoch over a small participant subset, so the goal is confirmation that artifacts are produced rather than strong performance. Expected outputs are the training-history table, the best epoch, and the output directory containing history, validation metrics, confusion matrix, plots, and checkpoints. The validation split is used for monitoring; the test split remains untouched.

In [6]:
if artifacts_available:
    training_result = train_model(loaders["train"], loaders["validation"], stage8_config)
    display(training_result.history)
    print("best epoch:", training_result.best_epoch)
    print("outputs:", training_result.output_dir)


,epoch,train_eval_ran,epoch_seconds,train_seconds,train_eval_seconds,validation_seconds,train_cache_loads,train_eval_cache_loads,validation_cache_loads,train_loss,...,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
0,1,True,44.155276,18.173381,9.349067,16.632686,0,0,3,0.621278,...,0.373079,0.24,0.28877,0.262136,0.786094,0.893033,0.836158,0.083333,0.011976,0.020942


best epoch: 1
outputs: /home/manns79/dreamt-wearable-sleep-staging/results/stage8_single_epoch_cnn


## Stage 9 Training-Choice Experiments

Stage 9 keeps the model family fixed to the single-epoch CNN and compares basic training choices using validation macro F1 as the primary selection metric. The first cell defines the Stage 9 base configuration and expands it into a controlled screening grid over unweighted versus train-only class-weighted loss, learning rate, dropout including `0.0`, and weight decay. The expected output is the number of configured runs. These configurations still use only the train and validation splits; the test split remains untouched.

In [7]:
stage9_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage9_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=10,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage9_screening_configs = build_stage9_screening_configs(
    base_config=stage9_base_config,
    output_dir=stage9_output_dir,
    learning_rates=(1e-4, 3e-4, 1e-3),
    dropouts=(0.0, 0.10),
    weight_decays=(0.0,),
    class_weighting_options=(True,),
    batch_sizes=(32,),
)
len(stage9_screening_configs)


6

The full default grid has 54 runs. For a quick local dry run, slice `stage9_screening_configs` before calling `run_stage9_experiments`; for the main Stage 9 result, run the full list on the local machine/GPU. The next cell is guarded by `RUN_STAGE9_EXPERIMENTS = False` so routine notebook execution does not accidentally launch a long training sweep. When enabled, the expected output is a top-10 validation summary sorted by macro F1 plus a results directory containing per-run histories, validation metrics, confusion matrices, checkpoints, and aggregate Stage 9 summary files.

In [8]:
RUN_STAGE9_EXPERIMENTS = False

if artifacts_available and RUN_STAGE9_EXPERIMENTS:
    stage9_summary = run_stage9_experiments(
        stage9_screening_configs,
        output_dir=stage9_output_dir,
    )
    display(stage9_summary.sort_values("macro_f1", ascending=False).head(10))
    print("outputs:", stage9_output_dir)
else:
    print("Stage 9 experiments are configured but not run in this notebook execution.")


Stage 9 experiments are configured but not run in this notebook execution.


## Stage 10 Temporal-Context CNN Comparison

Stage 10 asks whether neighboring epochs improve validation performance relative to the simple 1D CNN. The comparison is deliberately conservative: it keeps the CNN architecture and broad training defaults fixed, runs only `context_radius=1` and `context_radius=2`, and pairs each context CNN with a simple CNN trained and evaluated on the same context-eligible center epochs. This means the validation comparison is between center-only input and center-plus-neighbor input, not between different validation epoch sets. The test split remains untouched.


In [9]:
stage10_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage10_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    dropout=0.0,
    class_weighting=True,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage10_comparison_configs = build_stage10_comparison_configs(
    base_config=stage10_base_config,
    output_dir=stage10_output_dir,
    context_radii=(2, 5),
)
[(config.model_name, config.context_radius, config.comparison_context_radius) for config in stage10_comparison_configs]


[('single_epoch_cnn_stage10_context_eligible_r2', 0, 2),
 ('context_cnn_stage10_r2', 2, 2),
 ('single_epoch_cnn_stage10_context_eligible_r5', 0, 5),
 ('context_cnn_stage10_r5', 5, 5)]

The next cell is guarded by `RUN_STAGE10_EXPERIMENTS = False`. When enabled, it trains four validation-only runs: simple CNN and context CNN for radius 1, then simple CNN and context CNN for radius 2. The output summary includes validation macro F1, balanced accuracy, class-level metrics, the paired context radius, and the number of matched center epochs used for each comparison. Broader hyperparameter tuning is intentionally deferred to Stage 11.


In [10]:
RUN_STAGE10_EXPERIMENTS = False

if artifacts_available and RUN_STAGE10_EXPERIMENTS:
    stage10_summary = run_stage10_experiments(
        stage10_comparison_configs,
        output_dir=stage10_output_dir,
    )
    display(stage10_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage10_output_dir)
else:
    print("Stage 10 experiments are configured but not run in this notebook execution.")


Stage 10 experiments are configured but not run in this notebook execution.


## Stage 11 CNN-GRU Sequence Comparison

Stage 11 trains the first recurrent deep-learning model: a many-to-one CNN-GRU. Each epoch in a consecutive sequence is encoded by the 1D CNN trunk, the bidirectional GRU models temporal context around the center epoch, and the classifier predicts that center label. The initial run uses `sequence_length=5` with class-weighted cross entropy. Balanced sampling is intentionally not used so participant-block loading remains cache-friendly. The test split remains untouched.


In [11]:
stage11_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage11_output_dir,
    channels=CHANNELS,
    batch_size=16,
    epochs=25,
    patience=5,
    learning_rate=3e-4,
    weight_decay=1e-4,
    filters=(16, 32, 64),
    kernel_size=31,
    dropout=0.0,
    class_weighting=True,
    max_grad_norm=1.0,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    sequence_stride=1,
    sequence_label_mode="many_to_one",
    sequence_target_position="center",
    gru_hidden_size=64,
    gru_num_layers=1,
    gru_dropout=0.0,
    gru_bidirectional=True,
)

stage11_sequence_configs = build_stage11_sequence_configs(
    base_config=stage11_base_config,
    output_dir=stage11_output_dir,
    sequence_lengths=(5,),
)
[(config.model_name, config.sequence_length, config.sequence_target_position, config.class_weighting) for config in stage11_sequence_configs]


[('cnn_gru_stage11_s5', 5, 'center', True)]

The next cell is guarded by `RUN_STAGE11_EXPERIMENTS = False`. When enabled, it trains the initial `sequence_length=5` many-to-one CNN-GRU. The output summary includes validation macro F1, balanced accuracy, class-level metrics, and the number of sequence examples used for train and validation.


In [12]:
RUN_STAGE11_EXPERIMENTS = False

if artifacts_available and RUN_STAGE11_EXPERIMENTS:
    stage11_summary = run_stage11_experiments(
        stage11_sequence_configs,
        output_dir=stage11_output_dir,
    )
    display(stage11_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage11_output_dir)
else:
    print("Stage 11 experiments are configured but not run in this notebook execution.")


Stage 11 experiments are configured but not run in this notebook execution.


## Stage 11 Class-Prior Diagnostic

This no-training diagnostic tests whether the completed weighted-loss model learned useful class separation but an overly REM-heavy decision boundary. It multiplies saved validation probabilities by powered training priors for `alpha` values from 0 through 1, then saves validation metrics for each correction. These results are development diagnostics on the validation set and are not final unbiased performance estimates.


In [13]:
stage11_prior_correction_summary = pd.DataFrame()
stage11_summary_path = stage11_output_dir / "experiment_summary.csv"

if stage11_summary_path.exists():
    completed_stage11 = pd.read_csv(stage11_summary_path).sort_values(
        "macro_f1", ascending=False
    )
    completed_run_dir = Path(completed_stage11.iloc[0]["output_dir"])
    prediction_path = completed_run_dir / "validation_epoch_predictions.csv"
    config_path = completed_run_dir / "config.json"
    if prediction_path.exists() and config_path.exists():
        completed_config = load_train_config(config_path=config_path)
        completed_datasets = build_train_validation_datasets(completed_config)
        count_loader = torch.utils.data.DataLoader(
            completed_datasets["train"], batch_size=completed_config.batch_size
        )
        stage11_train_counts = class_counts_from_loader(count_loader)
        stage11_predictions = pd.read_csv(prediction_path)
        stage11_prior_correction_summary = class_prior_correction_sweep(
            stage11_predictions,
            stage11_train_counts,
            alphas=(0.0, 0.25, 0.5, 0.75, 1.0),
        )
        stage11_prior_correction_summary.to_csv(
            stage11_output_dir / "prior_correction_summary.csv", index=False
        )

display(stage11_prior_correction_summary)


,alpha,n_predictions,train_prior_Wake,train_prior_Non_REM,train_prior_REM,predicted_fraction_Wake,predicted_fraction_Non_REM,predicted_fraction_REM,accuracy,balanced_accuracy,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
0,1.00,11963,0.270923,0.627488,0.101589,0.209061,0.705676,0.085263,0.585054,0.374894,0.368753,0.268293,0.324155,0.293590,0.738688,0.735464,0.737072,0.090196,0.065064,0.075596
1,0.75,11963,0.270923,0.627488,0.101589,0.233052,0.447547,0.319401,0.434339,0.384696,0.352714,0.265423,0.357488,0.304652,0.746358,0.471282,0.577749,0.120387,0.325318,0.175740
2,0.50,11963,0.270923,0.627488,0.101589,0.247012,0.349494,0.403494,0.380172,0.385650,0.332669,0.263283,0.375845,0.309652,0.765128,0.377285,0.505371,0.118293,0.403819,0.182983
3,0.25,11963,0.270923,0.627488,0.101589,0.255454,0.273259,0.471286,0.333863,0.390194,0.311372,0.261453,0.385990,0.311744,0.765678,0.295200,0.426115,0.122739,0.489392,0.196256
4,0.00,11963,0.270923,0.627488,0.101589,0.262225,0.200284,0.537491,0.287303,0.392731,0.284394,0.261077,0.395652,0.314577,0.757095,0.213940,0.333609,0.125039,0.568600,0.204997


## Stage 11 Loss-Weighting Follow-Up

The follow-up keeps the five-epoch bidirectional CNN-GRU fixed and compares only two loss choices: unweighted cross entropy and square-root inverse-frequency weighting. Both runs use a 15-epoch cap and patience 4, and write to a separate output directory so the completed Stage 11 run is preserved.


In [14]:
stage11_loss_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage11_loss_output_dir,
    channels=CHANNELS,
    batch_size=16,
    epochs=15,
    patience=4,
    learning_rate=3e-4,
    weight_decay=1e-4,
    filters=(16, 32, 64),
    kernel_size=31,
    dropout=0.0,
    max_grad_norm=1.0,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    sequence_stride=1,
    sequence_label_mode="many_to_one",
    sequence_target_position="center",
    gru_hidden_size=64,
    gru_num_layers=1,
    gru_dropout=0.0,
    gru_bidirectional=True,
)

stage11_loss_configs = build_stage11_loss_comparison_configs(
    base_config=stage11_loss_base_config,
    output_dir=stage11_loss_output_dir,
    sequence_length=5,
)
[(config.model_name, config.class_weighting, config.class_weight_power) for config in stage11_loss_configs]


[('cnn_gru_stage11_s5_unweighted', False, 1.0),
 ('cnn_gru_stage11_s5_sqrt_weighted', True, 0.5)]

In [15]:
RUN_STAGE11_LOSS_EXPERIMENTS = False

if artifacts_available and RUN_STAGE11_LOSS_EXPERIMENTS:
    stage11_loss_summary = run_stage11_experiments(
        stage11_loss_configs,
        output_dir=stage11_loss_output_dir,
    )
    display(stage11_loss_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage11_loss_output_dir)
else:
    print("Stage 11 loss experiments are configured but not run.")


Stage 11 loss experiments are configured but not run.


## Stage 12 Many-To-Many CNN-GRU Aggregation

Stage 12 trains CNN-GRU models that predict a label for every epoch in each input sequence. Because overlapping sequences produce multiple probability predictions for most sleep epochs, validation aggregates those probabilities back to one prediction per sleep epoch before model selection. The default setup compares `sequence_length=5` and `sequence_length=11`, uses class-weighted cross entropy, weights each sequence-position loss by `1 / number_of_times_that_sleep_epoch_appears`, and evaluates both uniform probability averaging and center-weighted probability averaging from the same trained checkpoint. Balanced sampling is intentionally not used. The test split remains untouched.


In [16]:
stage12_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage12_output_dir,
    channels=CHANNELS,
    batch_size=16,
    epochs=12,
    patience=8,
    learning_rate=3e-4,
    weight_decay=1e-4,
    filters=(16, 32, 64),
    kernel_size=31,
    dropout=0.0,
    class_weighting=True,
    max_grad_norm=1.0,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    sequence_stride=1,
    sequence_label_mode="many_to_many",
    sequence_target_position="center",
    sequence_loss_weighting="inverse_epoch_coverage",
    gru_hidden_size=64,
    gru_num_layers=1,
    gru_dropout=0.0,
    gru_bidirectional=False,
)

stage12_many_to_many_configs = build_stage12_many_to_many_configs(
    base_config=stage12_base_config,
    output_dir=stage12_output_dir,
    sequence_lengths=(5, 11),
    aggregation_methods=("uniform", "center_weighted"),
)
[(config.model_name, config.sequence_length, config.sequence_aggregation, config.sequence_extra_aggregations, config.sequence_loss_weighting) for config in stage12_many_to_many_configs]


[('cnn_gru_stage12_m2m_s5',
  5,
  'uniform',
  ('center_weighted',),
  'inverse_epoch_coverage'),
 ('cnn_gru_stage12_m2m_s11',
  11,
  'uniform',
  ('center_weighted',),
  'inverse_epoch_coverage')]

The next cell is guarded by `RUN_STAGE12_EXPERIMENTS = False`. When enabled, it trains one many-to-many CNN-GRU per sequence length, then reports one summary row per aggregation method. The output includes raw sequence-position metrics, aggregated per-sleep-epoch metrics, per-position validation predictions, aggregated epoch-level validation predictions, confusion matrices, and the usual training histories/checkpoints.


In [17]:
RUN_STAGE12_EXPERIMENTS = False

if artifacts_available and RUN_STAGE12_EXPERIMENTS:
    stage12_summary = run_stage12_experiments(
        stage12_many_to_many_configs,
        output_dir=stage12_output_dir,
    )
    display(stage12_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage12_output_dir)
else:
    print("Stage 12 experiments are configured but not run in this notebook execution.")


Stage 12 experiments are configured but not run in this notebook execution.


## Stage 14 Multiscale Residual Feature-Fusion CNN

Stage 14 replaces the underpowered single-vector epoch encoder with three raw-signal convolution scales, GroupNorm residual blocks, and 12 retained temporal bins. A compact MLP encodes the 72 engineered Stage 6 features, and the two embeddings are fused for one three-class prediction per epoch. Raw-signal and engineered-feature preprocessing are fit on training rows only using memory-efficient means and standardization; validation features reuse the saved training metadata. The fixed run uses unweighted cross-entropy with label smoothing, validation macro F1 checkpointing, gradient clipping, and final/stopping-epoch train evaluation.

In [18]:
stage14_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage14_output_dir,
    channels=CHANNELS,
    train_feature_path=train_feature_path,
    validation_feature_path=validation_feature_path,
    feature_preprocessing_metadata_path=feature_metadata_path,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage14_config = build_stage14_fusion_config(
    base_config=stage14_base_config,
    output_dir=stage14_output_dir,
)
stage14_config


TrainConfig(raw_dir=PosixPath('/home/manns79/dreamt-wearable-sleep-staging/data/raw'), epoch_index_path=PosixPath('/home/manns79/dreamt-wearable-sleep-staging/data/interim/epoch_index.csv'), preprocessing_metadata_path=PosixPath('/home/manns79/dreamt-wearable-sleep-staging/data/processed/preprocessing_metadata.json'), output_dir=PosixPath('/home/manns79/dreamt-wearable-sleep-staging/results/stage14_multiscale_fusion_cnn'), channels=['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI'], model_name='multiscale_residual_fusion_cnn_stage14', model_type='multiscale_fusion', batch_size=32, epochs=25, learning_rate=0.0003, weight_decay=0.0001, filters=(16, 32, 64), kernel_size=31, dropout=0.1, context_radius=0, comparison_context_radius=None, sequence_length=1, sequence_stride=1, sequence_label_mode='many_to_one', sequence_target_position='last', sequence_loss_weighting='none', sequence_aggregation='none', sequence_extra_aggregations=(), gru_hidden_size=64, gru_num_layers=1, gru_drop

The launch cell is deliberately guarded by `RUN_STAGE14_EXPERIMENT = False`. Enabling it runs the one fixed Stage 14 configuration; there is no sweep. Expected outputs include the run config, train/validation histories, macro-F1-selected checkpoints, validation predictions and confusion matrix, plus `results/stage14_multiscale_fusion_cnn/experiment_summary.csv`.

In [19]:
RUN_STAGE14_EXPERIMENT = True

if stage14_artifacts_available and RUN_STAGE14_EXPERIMENT:
    stage14_summary = run_stage14_experiment(
        stage14_config,
        output_dir=stage14_output_dir,
    )
    display(stage14_summary)
    print("outputs:", stage14_output_dir)
else:
    print("Stage 14 is configured but not run in this notebook execution.")


,experiment_id,model_family,train_examples,validation_examples,best_epoch,output_dir,raw_dir,epoch_index_path,preprocessing_metadata_path,channels,...,macro_f1,Wake_precision,Wake_recall,Wake_f1,Non_REM_precision,Non_REM_recall,Non_REM_f1,REM_precision,REM_recall,REM_f1
0,stage14_multiscale_fusion_0fb086beb5,multiscale_residual_fusion,55793,12027,11,/home/manns79/dreamt-wearable-sleep-staging/re...,/home/manns79/dreamt-wearable-sleep-staging/da...,/home/manns79/dreamt-wearable-sleep-staging/da...,/home/manns79/dreamt-wearable-sleep-staging/da...,"[BVP, ACC_X, ACC_Y, ACC_Z, TEMP, EDA, HR, IBI]",...,0.421734,0.469633,0.315166,0.377198,0.7371,0.865937,0.796341,0.149206,0.066151,0.091663


outputs: /home/manns79/dreamt-wearable-sleep-staging/results/stage14_multiscale_fusion_cnn
